In [0]:
from pyspark.sql import functions as F

DB = "workspace.review_analytics"
raw = spark.table(f"{DB}.reviews_all")
print(f"Loaded {raw.count():,} reviews")

Loaded 1,140,000 reviews


In [0]:
# Common English stop words, minus negations ("not", "no", "never"...),
# because "not good" and "good" mean opposite things for sentiment.
STOP_WORDS = """a an the and or but if then so of to in on at by for with from as is are was
were be been being am it its it's this that these those i me my we our you your he she him her
they them their what which who whom there here when where why how all any both each few more most
other some such only own same than too very can will just do does did doing have has had having
would should could also get got one would about into over after before up down out off again
further once s t ve ll re d m""".split()

clean = (raw
    # 1. Combine title + review text (titles carry strong sentiment, e.g. "Total junk!")
    .withColumn("review_text", F.concat_ws(". ", "title", "text"))
    # 2. Lowercase
    .withColumn("clean_text", F.lower("review_text"))
    # 3. Remove HTML tags like <br/>
    .withColumn("clean_text", F.regexp_replace("clean_text", r"<[^>]+>", " "))
    # 4. Remove URLs
    .withColumn("clean_text", F.regexp_replace("clean_text", r"(https?://\S+|www\.\S+)", " "))
    # 5. Expand "n't" -> " not"  (don't -> do not)
    .withColumn("clean_text", F.regexp_replace("clean_text", r"n't\b", " not"))
    # 6. Keep only letters and spaces
    .withColumn("clean_text", F.regexp_replace("clean_text", r"[^a-z\s]", " "))
    # 7. Collapse repeated spaces
    .withColumn("clean_text", F.trim(F.regexp_replace("clean_text", r"\s+", " ")))
    .filter(F.length("clean_text") > 0)
    .dropDuplicates(["review_id"])

    # Tokens: split into words, drop stop words and 1-letter words (used for word clouds/analytics)
    .withColumn("tokens", F.filter(F.split("clean_text", " "),
                                   lambda w: (F.length(w) > 1) & (~w.isin(STOP_WORDS))))
    .withColumn("word_count", F.size("tokens"))

    # Rating-derived proxy labels
    .withColumn("sentiment", F.when(F.col("rating") <= 2, "Negative")
                              .when(F.col("rating") == 3, "Neutral")
                              .otherwise("Positive"))
    .withColumn("label", F.when(F.col("rating") <= 2, 0.0)
                          .when(F.col("rating") == 3, 1.0)
                          .otherwise(2.0))
    .withColumn("review_month", F.substring("review_date", 1, 7))   # e.g. 2024-03
)
print("Cleaning plan built (nothing has run yet - Spark is lazy)")

Cleaning plan built (nothing has run yet - Spark is lazy)


In [0]:
(clean.write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{DB}.reviews_clean"))

reviews_clean = spark.table(f"{DB}.reviews_clean")
print(f"reviews_clean: {reviews_clean.count():,} rows")

reviews_clean: 1,139,264 rows


In [0]:
display(reviews_clean.select("rating", "sentiment", "title", "clean_text", "tokens").limit(10))

rating,sentiment,title,clean_text,tokens
2.0,Negative,"Meh … not comfortable, short battery life",meh not comfortable short battery life i was hoping these could replace my bf sleeping with air pods in but the battery life was terrible and the hard discs near the ears hurt him since he is a side sleeper oh well we tried,"List(meh, not, comfortable, short, battery, life, hoping, replace, bf, sleeping, air, pods, battery, life, terrible, hard, discs, near, ears, hurt, since, side, sleeper, oh, well, tried)"
5.0,Positive,worth buying,worth buying price was good good value for the money,"List(worth, buying, price, good, good, value, money)"
1.0,Negative,Wish I hadn't purchased,wish i had not purchased for the life of me i ca not get this to play a dvd,"List(wish, not, purchased, life, ca, not, play, dvd)"
5.0,Positive,Works great with older OS like Windows 7,works great with older os like windows i use it as an external backup device for my aging desktop pc running on windows some newer drives do not support this vintage os took about minutes to initialize the drive by launching the enclosed exe that goes out to wd site for the actual executable for install everything works great and the ui is exactly like the traditional windows explorer i was able to either drag drop the data from my pc to this drive just like from one folder to another within the pc but i actually used the backup tool so that i can perform incremental backups the drive also comes with system diagnostic tool that can be handy for troubleshooting not that i necessarily have the skill to proficiently read and understand all the commands and syntaxes from the reports but i am sure it would be useful for those who are smart enough to do such things it s lightweight and the footprint is small as advertised not important to me since i do not plan on carrying it around but it would be great for those who do as its true pocket size should be able to fit inside and jacket pockets easily,"List(works, great, older, os, like, windows, use, external, backup, device, aging, desktop, pc, running, windows, newer, drives, not, support, vintage, os, took, minutes, initialize, drive, launching, enclosed, exe, goes, wd, site, actual, executable, install, everything, works, great, ui, exactly, like, traditional, windows, explorer, able, either, drag, drop, data, pc, drive, like, folder, another, within, pc, actually, used, backup, tool, perform, incremental, backups, drive, comes, system, diagnostic, tool, handy, troubleshooting, not, necessarily, skill, proficiently, read, understand, commands, syntaxes, reports, sure, useful, smart, enough, things, lightweight, footprint, small, advertised, not, important, since, not, plan, carrying, around, great, true, pocket, size, able, fit, inside, jacket, pockets, easily)"
5.0,Positive,Great buy,great buy i like the sturdiness and variance of length in the order,"List(great, buy, like, sturdiness, variance, length, order)"
5.0,Positive,Perfect,perfect perfect for connecting my phone to my charging block without a lot of excess cord love it,"List(perfect, perfect, connecting, phone, charging, block, without, lot, excess, cord, love)"
5.0,Positive,So far so good!,so far so good i use this to power my macbook it s been working perfectly,"List(far, good, use, power, macbook, working, perfectly)"
5.0,Positive,So far great,so far great so far is very good i use it for my editing and voice over and so far it great but let us see in a few months is really fast by the way,"List(far, great, far, good, use, editing, voice, far, great, let, us, see, months, really, fast, way)"
3.0,Neutral,Quality as expected,quality as expected this camera does record and capture well there are a few features that are enabled at start that interrupt recording if not disabled it comes with allot of useful accessories and attachments but is kinda fragile it dropped from a feet and the battery door broke,"List(quality, expected, camera, record, capture, well, features, ena

In [0]:
from pyspark.ml.feature import Tokenizer, HashingTF

test = spark.createDataFrame([("this phone is not bad at all",)], ["text"])
words = Tokenizer(inputCol="text", outputCol="words").transform(test)
HashingTF(inputCol="words", outputCol="tf", numFeatures=32).transform(words).show(truncate=False)
print("MLlib works on this workspace")

+----------------------------+------------------------------------+-----------------------------------+
|text                        |words                               |tf                                 |
+----------------------------+------------------------------------+-----------------------------------+
|this phone is not bad at all|[this, phone, is, not, bad, at, all]|(32,[4,10,25,29],[2.0,2.0,1.0,2.0])|
+----------------------------+------------------------------------+-----------------------------------+

MLlib works on this workspace
